# Statistical Analysis — Speech-to-Text Evaluation
**Metrics:** WER, CER, SER, MKA  
**Tests:** Descriptive Stats · F1 · ANOVA (F-statistic) · Tukey HSD · Wilcoxon Signed-Rank (pre vs post)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import f_oneway, wilcoxon, shapiro, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd

matplotlib.rcParams['font.family'] = 'Tahoma'
pd.set_option('display.float_format', '{:.4f}'.format)

EXCEL_PATH = r'C:\code\git\Senior_project\khaiwan\ตารางเปรียบเทียบการแปลงเสียง.xlsx'
METRICS    = ['WER', 'CER', 'SER', 'MKA']
MODELS     = ['whisper', 'elevenlab', 'soniox']

print('Libraries loaded ✓')

## 1 · Load & Clean Data

In [ ]:
def clean_numeric(val):
    """Strip % and whitespace, return float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace('%', '').replace(',', '.')
    try:
        return float(s)
    except:
        return np.nan

def load_sheet(excel_path, sheet_name, model_label, phase):
    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=0)
    df.columns = ['filename', 'source', 'time_min', 'WER', 'CER', 'SER', 'MKA']
    for m in METRICS:
        df[m] = df[m].apply(clean_numeric)
    df['time_min'] = df['time_min'].apply(clean_numeric)
    df = df.dropna(subset=['WER'])
    df['model'] = model_label
    df['phase'] = phase
    return df

sheets = {
    'pre_whisper':   ('whisper',   'pre'),
    'pre_elevenlab': ('elevenlab', 'pre'),
    'pre_soniox':    ('soniox',    'pre'),
    'post_whisper':  ('whisper',   'post'),
    'post_elevenlab':('elevenlab', 'post'),
    'post_soniox':   ('soniox',    'post'),
}

frames = [load_sheet(EXCEL_PATH, sn, label, phase)
          for sn, (label, phase) in sheets.items()]
df_all = pd.concat(frames, ignore_index=True)

print('Total rows:', len(df_all))
print(df_all.groupby(['phase', 'model']).size().unstack())

## 2 · Descriptive Statistics (Mean ± SD, Median, Min, Max)

In [ ]:
def descriptive(df):
    return df.groupby(['phase', 'model'])[METRICS].agg(['mean','std','median','min','max']).round(4)

desc = descriptive(df_all)
print('=== Descriptive Statistics ===')
display(desc)

In [ ]:
# Clean summary table: mean ± std per metric
rows = []
for phase in ['pre', 'post']:
    for model in MODELS:
        sub = df_all[(df_all.phase == phase) & (df_all.model == model)]
        row = {'Phase': phase, 'Model': model, 'N': len(sub)}
        for m in METRICS:
            row[m] = f"{sub[m].mean():.2f} ± {sub[m].std():.2f}"
        rows.append(row)

summary_table = pd.DataFrame(rows)
print('=== Mean ± SD Summary ===')
display(summary_table)

## 3 · F1 Score Calculation
F1 (ASR) = harmonic mean of Word Accuracy (1 − WER/100) and MKA score (MKA/100)

In [ ]:
def compute_f1(wer_pct, mka_pct):
    """F1 = 2*WA*MKA_norm / (WA + MKA_norm)"""
    wa  = 1.0 - wer_pct / 100.0   # Word Accuracy
    mka = mka_pct / 100.0
    denom = wa + mka
    return np.where(denom > 0, 2 * wa * mka / denom, 0.0)

df_all['F1'] = compute_f1(df_all['WER'], df_all['MKA'])

f1_summary = df_all.groupby(['phase', 'model'])['F1'].agg(['mean','std','median']).round(4)
print('=== F1 Score (mean ± std) ===')
display(f1_summary)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, phase in zip(axes, ['pre', 'post']):
    vals = df_all[df_all.phase == phase].groupby('model')['F1'].mean()
    errs = df_all[df_all.phase == phase].groupby('model')['F1'].std()
    ax.bar(vals.index, vals.values, yerr=errs.values, capsize=5,
           color=['#4472C4','#ED7D31','#A9D18E'])
    ax.set_title(f'F1 Score — {phase.upper()}')
    ax.set_ylabel('F1'); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 4 · Normality Test (Shapiro-Wilk)
ใช้ตัดสินว่าควรใช้ parametric (ANOVA) หรือ non-parametric (Kruskal-Wallis)

In [ ]:
normality_rows = []
for phase in ['pre', 'post']:
    for model in MODELS:
        sub = df_all[(df_all.phase == phase) & (df_all.model == model)]
        for m in METRICS + ['F1']:
            data = sub[m].dropna().values
            if len(data) >= 3:
                stat, p = shapiro(data)
                normality_rows.append({
                    'Phase': phase, 'Model': model, 'Metric': m,
                    'W': round(stat, 4), 'p-value': round(p, 4),
                    'Normal (p>0.05)': 'Yes' if p > 0.05 else 'No'
                })

norm_df = pd.DataFrame(normality_rows)
print('=== Shapiro-Wilk Normality Test ===')
display(norm_df)

## 5 · One-Way ANOVA — Compare 3 Algorithms (F-statistic + p-value)

In [ ]:
anova_rows = []
for phase in ['pre', 'post']:
    for m in METRICS + ['F1']:
        groups = [
            df_all[(df_all.phase == phase) & (df_all.model == mod)][m].dropna().values
            for mod in MODELS
        ]
        # ANOVA (parametric)
        f_stat, p_anova = f_oneway(*groups)
        # Kruskal-Wallis (non-parametric alternative)
        h_stat, p_kruskal = kruskal(*groups)
        anova_rows.append({
            'Phase': phase, 'Metric': m,
            'F-statistic': round(f_stat, 4),
            'p-value (ANOVA)': round(p_anova, 6),
            'Sig (ANOVA)': '***' if p_anova<0.001 else ('**' if p_anova<0.01 else ('*' if p_anova<0.05 else 'ns')),
            'H-statistic (KW)': round(h_stat, 4),
            'p-value (KW)': round(p_kruskal, 6),
            'Sig (KW)': '***' if p_kruskal<0.001 else ('**' if p_kruskal<0.01 else ('*' if p_kruskal<0.05 else 'ns')),
        })

anova_df = pd.DataFrame(anova_rows)
print('=== One-Way ANOVA + Kruskal-Wallis (Algorithm Comparison) ===')
print('Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
display(anova_df)

## 6 · Tukey HSD Post-Hoc Test (Pairwise Algorithm Comparison)

In [ ]:
for phase in ['pre', 'post']:
    print(f'\n{'='*60}')
    print(f'  TUKEY HSD — {phase.upper()}')
    print(f'{'='*60}')
    for m in METRICS + ['F1']:
        sub = df_all[df_all.phase == phase][['model', m]].dropna()
        if len(sub) < 6:
            continue
        tukey = pairwise_tukeyhsd(endog=sub[m], groups=sub['model'], alpha=0.05)
        tukey_df = pd.DataFrame(
            data=tukey._results_table.data[1:],
            columns=tukey._results_table.data[0]
        )
        tukey_df['meandiff'] = tukey_df['meandiff'].astype(float).round(4)
        tukey_df['p-adj']    = tukey_df['p-adj'].astype(float).round(6)
        tukey_df['lower']    = tukey_df['lower'].astype(float).round(4)
        tukey_df['upper']    = tukey_df['upper'].astype(float).round(4)
        print(f'\n  Metric: {m}')
        display(tukey_df)

## 7 · Wilcoxon Signed-Rank Test — Pre vs Post (each Algorithm)

In [ ]:
wilcox_rows = []
for model in MODELS:
    for m in METRICS + ['F1']:
        pre_vals  = df_all[(df_all.phase == 'pre')  & (df_all.model == model)][m].reset_index(drop=True)
        post_vals = df_all[(df_all.phase == 'post') & (df_all.model == model)][m].reset_index(drop=True)

        # align length
        n = min(len(pre_vals), len(post_vals))
        pre_arr  = pre_vals.iloc[:n].values
        post_arr = post_vals.iloc[:n].values

        # drop pairs where diff == 0 (wilcoxon requirement)
        mask = pre_arr != post_arr
        if mask.sum() < 3:
            continue

        stat, p = wilcoxon(pre_arr[mask], post_arr[mask])

        # Effect size: r = Z / sqrt(N)
        z = stats.norm.ppf(p / 2)
        r = abs(z) / np.sqrt(mask.sum())

        mean_pre  = pre_arr.mean()
        mean_post = post_arr.mean()
        direction = 'improved' if (m in ['MKA','F1'] and mean_post > mean_pre) or \
                                  (m not in ['MKA','F1'] and mean_post < mean_pre) else 'degraded'

        wilcox_rows.append({
            'Model': model, 'Metric': m,
            'Mean_pre': round(mean_pre, 4), 'Mean_post': round(mean_post, 4),
            'Direction': direction,
            'W-statistic': round(stat, 4),
            'p-value': round(p, 6),
            'Effect size (r)': round(r, 4),
            'Sig': '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
        })

wilcox_df = pd.DataFrame(wilcox_rows)
print('=== Wilcoxon Signed-Rank Test (Pre vs Post) ===')
print('Direction: improved = post is better than pre')
print('Significance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
display(wilcox_df)

## 8 · Visualization — Box Plots per Metric

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, len(METRICS), figsize=(16, 10))
colors = {'whisper': '#4472C4', 'elevenlab': '#ED7D31', 'soniox': '#A9D18E'}

for col, m in enumerate(METRICS):
    for row, phase in enumerate(['pre', 'post']):
        ax = axes[row][col]
        data_by_model = [
            df_all[(df_all.phase == phase) & (df_all.model == mod)][m].dropna().values
            for mod in MODELS
        ]
        bp = ax.boxplot(data_by_model, patch_artist=True, notch=False,
                        medianprops=dict(color='black', linewidth=2))
        for patch, mod in zip(bp['boxes'], MODELS):
            patch.set_facecolor(colors[mod])
        ax.set_title(f'{m} — {phase.upper()}', fontsize=10)
        ax.set_xticklabels(['Whisper', 'ElevenLab', 'Soniox'], rotation=15)
        ax.set_ylabel('%')
        ax.grid(axis='y', alpha=0.3)

legend_handles = [mpatches.Patch(color=c, label=m) for m, c in colors.items()]
fig.legend(handles=legend_handles, loc='lower center', ncol=3, fontsize=10)
fig.suptitle('Metric Distribution by Algorithm & Phase', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 9 · Export All Results to Excel

In [ ]:
out_stat = r'C:\code\git\Senior_project\khaiwan\stat_results.xlsx'

with pd.ExcelWriter(out_stat, engine='openpyxl') as writer:
    summary_table.to_excel(writer, sheet_name='Mean_SD_Summary', index=False)
    norm_df.to_excel(writer, sheet_name='Shapiro_Wilk', index=False)
    anova_df.to_excel(writer, sheet_name='ANOVA_KruskalWallis', index=False)
    wilcox_df.to_excel(writer, sheet_name='Wilcoxon_pre_vs_post', index=False)

    # Tukey HSD per phase+metric into one sheet
    tukey_all = []
    for phase in ['pre', 'post']:
        for m in METRICS + ['F1']:
            sub = df_all[df_all.phase == phase][['model', m]].dropna()
            if len(sub) < 6:
                continue
            tukey = pairwise_tukeyhsd(endog=sub[m], groups=sub['model'], alpha=0.05)
            t_df = pd.DataFrame(tukey._results_table.data[1:],
                                columns=tukey._results_table.data[0])
            t_df.insert(0, 'Phase', phase)
            t_df.insert(1, 'Metric', m)
            tukey_all.append(t_df)
    pd.concat(tukey_all, ignore_index=True).to_excel(writer, sheet_name='Tukey_HSD', index=False)

print('Exported:', out_stat)

## 10 · Quick Reference Summary

In [ ]:
print('='*65)
print('  STATISTICAL ANALYSIS SUMMARY')
print('='*65)
print()
print('ANOVA significant results (p<0.05):')
sig = anova_df[anova_df['Sig (ANOVA)'] != 'ns'][['Phase','Metric','F-statistic','p-value (ANOVA)','Sig (ANOVA)']]
print(sig.to_string(index=False))
print()
print('Wilcoxon significant pre→post improvements (p<0.05):')
sig_w = wilcox_df[(wilcox_df['Sig'] != 'ns') & (wilcox_df['Direction'] == 'improved')]
print(sig_w[['Model','Metric','Mean_pre','Mean_post','p-value','Effect size (r)','Sig']].to_string(index=False))